# Dịch JSON: Tiếng Anh → nhiều ngôn ngữ (NLLB-200)

**Cách dùng:**
1. Chạy Cell 1 (cài thư viện) và Cell 2 (load model — lần đầu mất vài phút).
2. Chạy Cell 3 để **upload file JSON** từ máy bạn (không cần kéo thả thủ công).
3. Ở Cell 4, chỉ cần liệt kê **mã ngôn ngữ** muốn dịch tới trong `TARGET_LANG_CODES`.
4. Chạy Cell 5 để dịch và tải kết quả về.

In [ ]:
# Cell 1: Cài đặt thư viện
!pip install -q transformers sentencepiece torch accelerate

In [ ]:
# Cell 2: Load model NLLB-200
# distilled-600M: nhẹ, chạy tốt trên Colab free T4.
# Cần chất lượng cao hơn (và GPU khỏe hơn) thì đổi thành "facebook/nllb-200-1.3B".
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}")

In [ ]:
# Cell 3: Upload file JSON tiếng Anh từ máy tính
from google.colab import files
import json

uploaded = files.upload()  # sẽ hiện nút "Choose Files" để chọn file từ máy
INPUT_PATH = list(uploaded.keys())[0]
print(f"Đã upload: {INPUT_PATH}")

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    source_data = json.load(f)
print("Đọc file thành công.")

In [ ]:
# Cell 4: Chọn ngôn ngữ nguồn / đích — chỉ cần ghi mã ngôn ngữ
SRC_LANG_CODE = "en"

# Chỉ cần liệt kê các mã ở đây, không cần biết mã FLORES-200 bên dưới.
# TARGET_LANG_CODES = ["de", "es", "fr", "it", "ja", "ko", "nl", "pt", "ru", "tr",
#                       "vi", "ar", "fa", "hi", "id", "iw", "km", "th", "uk", "zh"]

TARGET_LANG_CODES = ["vi"]

# Chỉ những KEY nằm trong danh sách này mới bị dịch (THAY THẾ bản gốc), mọi key khác giữ nguyên.
# Chỉ cần khai báo tên key (leaf), KHÔNG cần khai báo cả đường dẫn cha —
# đệ quy sẽ tự khớp key này dù nó nằm sâu bao nhiêu cấp (vd trong steps[].messages[].text).
# Chỉ cần khai báo full path nếu CÙNG 1 TÊN KEY mang 2 ý nghĩa khác nhau ở 2 vị trí khác nhau.
TRANSLATE_KEYS = {"text", "title", "question", "hint", "textSuccess", "placeholder", "label", "condition", "date", "name", "subject", "body", "location", "description", "caption", "commentText", "bio", "content", "notes" }

# Những key có giá trị là LIST OF STRING mà bạn muốn GIỮ NGUYÊN bản gốc tiếng Anh
# và CHỈ THÊM bản dịch vào cuối list (không thay thế). Ví dụ: "answer" chứa nhiều
# biến thể viết hoa/thường của "do not answer" -> giữ hết bản gốc, thêm bản dịch tiếng Việt.
APPEND_TRANSLATE_KEYS = {"answer"}

# Bảng ánh xạ mã ngôn ngữ thông thường -> mã FLORES-200 (dùng nội bộ bởi NLLB).
LANG_CODE_MAP = {
    "en": "eng_Latn",
    "de": "deu_Latn",
    "es": "spa_Latn",
    "fr": "fra_Latn",
    "it": "ita_Latn",
    "ja": "jpn_Jpan",
    "ko": "kor_Hang",
    "nl": "nld_Latn",
    "pt": "por_Latn",
    "ru": "rus_Cyrl",
    "tr": "tur_Latn",
    "vi": "vie_Latn",
    "ar": "arb_Arab",
    "fa": "pes_Arab",
    "hi": "hin_Deva",
    "id": "ind_Latn",
    "iw": "heb_Hebr",  # "iw" là mã cũ của tiếng Do Thái (Hebrew), NLLB dùng "heb_Hebr"
    "km": "khm_Khmr",
    "th": "tha_Thai",
    "uk": "ukr_Cyrl",
    "zh": "zho_Hans",  # Trung giản thể. Cần phồn thể thì đổi thành "zho_Hant"
}

missing = [c for c in TARGET_LANG_CODES if c not in LANG_CODE_MAP]
if missing:
    raise ValueError(f"Chưa có mapping FLORES-200 cho mã: {missing}. Thêm vào LANG_CODE_MAP ở trên.")

In [ ]:
# Cell 5: Dịch và tải kết quả về (nén thành 1 file zip, cấu trúc cases/<lang_code>/<tên_file_input>.json)
import os
import shutil
import time

# Đếm số chuỗi đã dịch + tổng thời gian gọi model, để tính thời gian trung bình mỗi ngôn ngữ.
_call_stats = {"count": 0, "seconds": 0.0}

def translate_text(text: str, src: str, tgt: str) -> str:
    if not text or not text.strip():
        return text
    t0 = time.perf_counter()
    tokenizer.src_lang = src
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt)
    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        num_beams=5,
        max_length=512,
    )
    result = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    _call_stats["count"] += 1
    _call_stats["seconds"] += time.perf_counter() - t0
    return result

def translate_list_append(lst, src: str, tgt: str):
    # Giữ nguyên toàn bộ giá trị gốc, chỉ thêm bản dịch (không trùng) vào cuối list.
    seen = {s.strip().lower() for s in lst if isinstance(s, str)}
    appended = []
    for s in lst:
        if isinstance(s, str) and s.strip():
            translated = translate_text(s, src, tgt)
            if translated.strip().lower() not in seen:
                appended.append(translated)
                seen.add(translated.strip().lower())
    return list(lst) + appended

def translate_json(obj, src: str, tgt: str, current_key: str = None):
    # current_key = key của obj tại cấp cha (None nếu obj là phần tử của list hoặc root)
    if isinstance(obj, dict):
        result = {}
        for k, v in obj.items():
            if k in APPEND_TRANSLATE_KEYS and isinstance(v, list):
                result[k] = translate_list_append(v, src, tgt)
            else:
                result[k] = translate_json(v, src, tgt, current_key=k)
        return result
    elif isinstance(obj, list):
        # list không có key riêng -> giữ nguyên current_key của cha (vd "messages": [...])
        return [translate_json(item, src, tgt, current_key=current_key) for item in obj]
    elif isinstance(obj, str):
        # chỉ dịch nếu key trực tiếp chứa string này nằm trong TRANSLATE_KEYS
        if current_key in TRANSLATE_KEYS:
            return translate_text(obj, src, tgt)
        return obj
    else:
        return obj

src_flores = LANG_CODE_MAP[SRC_LANG_CODE]
# Tên file input không có phần đuôi .json, dùng làm tên file trong từng thư mục ngôn ngữ.
input_basename = os.path.splitext(os.path.basename(INPUT_PATH))[0]

# Cấu trúc thư mục: /content/cases/<lang_code>/<input_basename>.json
cases_root = "/content/cases"
if os.path.exists(cases_root):
    shutil.rmtree(cases_root)

output_paths = []
for code in TARGET_LANG_CODES:
    tgt_flores = LANG_CODE_MAP[code]
    print(f"Đang dịch sang '{code}' ({tgt_flores})...")

    _call_stats["count"] = 0
    _call_stats["seconds"] = 0.0
    lang_t0 = time.perf_counter()
    translated = translate_json(source_data, src_flores, tgt_flores)
    lang_elapsed = time.perf_counter() - lang_t0

    out_dir = os.path.join(cases_root, code)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{input_basename}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(translated, f, ensure_ascii=False, indent=2)
    output_paths.append(out_path)

    avg = _call_stats["seconds"] / _call_stats["count"] if _call_stats["count"] else 0.0
    print(
        f"  -> Đã lưu: {out_path} | "
        f"{_call_stats['count']} chuỗi trong {lang_elapsed:.1f}s "
        f"(trung bình {avg:.2f}s/chuỗi)"
    )

# Nén toàn bộ thư mục cases/ thành 1 file zip rồi tải về -> khi giải nén sẽ ra
# đúng cấu trúc cases/<lang_code>/<input_basename>.json.
zip_base = f"/content/{input_basename}_translated"
zip_path = shutil.make_archive(zip_base, "zip", cases_root)
print(f"\nHoàn tất. Đang tải file nén: {zip_path}")
files.download(zip_path)

## Ghi chú

- **Chỉ cần sửa `TARGET_LANG_CODES`** ở Cell 4 để thêm/bớt ngôn ngữ — không cần đụng vào phần dịch ở Cell 5.
- **Thêm ngôn ngữ chưa có trong `LANG_CODE_MAP`**: tìm mã FLORES-200 tương ứng tại https://github.com/facebookresearch/flores/blob/main/flores200/README.md rồi thêm 1 dòng vào bảng.
- **`files.download()`** sẽ tự động tải từng file JSON kết quả về máy bạn qua trình duyệt sau khi dịch xong; nếu trình duyệt chặn multiple downloads, cho phép popup cho trang Colab.
- **Batch dịch**: script dịch từng string một, ổn với vài trăm entry. Với JSON hàng nghìn entry, nên gom string thành batch để tăng tốc.
- **Chất lượng vs tốc độ**: đổi `model_name` ở Cell 2 sang `facebook/nllb-200-1.3B` hoặc `-3.3B` nếu cần dịch chính xác hơn (cần GPU mạnh hơn, Colab Pro).